# AgriSmartAI — Notebook 12: Fertilizer Model Robustness & Generalization Evaluation
### Offline Measurement-Uncertainty Stress Testing & Boundary Recovery Analysis

## 1. Setup & Environment Resolution

In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'notebooks' else cwd
if project_root not in sys.path:
    sys.path.insert(0, project_root)


## 2. Load Dataset & Production Model Artifacts

In [2]:
dataset_path = os.path.join(project_root, 'datasets', 'fertilizer_recommendation.csv')
prod_model_path = os.path.join(project_root, 'models', 'fertilizer_recommendation_model.pkl')

df = pd.read_csv(dataset_path)

def map_fertilizer(row):
    n, p, k = row['Nitrogen_Level'], row['Phosphorus_Level'], row['Potassium_Level']
    ph = row['Soil_pH']
    if ph < 5.5: return 'Lime & Organic Compost'
    elif ph > 8.0: return 'Gypsum & Sulfur'
    elif n < 40 and p >= 40 and k >= 40: return 'Urea (High Nitrogen)'
    elif p < 40 and n >= 40 and k >= 40: return 'DAP (Di-Ammonium Phosphate)'
    elif k < 40 and n >= 40 and p >= 40: return 'MOP (Muriate of Potash)'
    elif n < 50 and p < 50: return 'NPK 28-28-0 Complex'
    else: return 'NPK 17-17-17 Balanced'

if 'Fertilizer_Name' not in df.columns:
    df['Fertilizer_Name'] = df.apply(map_fertilizer, axis=1)

prod_model = joblib.load(prod_model_path)
print('Dataset shape:', df.shape)
print('Production Model Class:', type(prod_model).__name__)


Dataset shape: (10000, 11)
Production Model Class: DecisionTreeClassifier


## 3. Standard Evaluation (Reproducing 100.00% Baseline)

In [3]:
def map_fertilizer(row):
    n, p, k = row['Nitrogen_Level'], row['Phosphorus_Level'], row['Potassium_Level']
    ph = row['Soil_pH']
    if ph < 5.5: return 'Lime & Organic Compost'
    elif ph > 8.0: return 'Gypsum & Sulfur'
    elif n < 40 and p >= 40 and k >= 40: return 'Urea (High Nitrogen)'
    elif p < 40 and n >= 40 and k >= 40: return 'DAP (Di-Ammonium Phosphate)'
    elif k < 40 and n >= 40 and p >= 40: return 'MOP (Muriate of Potash)'
    elif n < 50 and p < 50: return 'NPK 28-28-0 Complex'
    else: return 'NPK 17-17-17 Balanced'

if 'Fertilizer_Name' not in df.columns:
    df['Fertilizer_Name'] = df.apply(map_fertilizer, axis=1)

df_encoded = pd.get_dummies(df, columns=['Soil_Type'], drop_first=True)
X = df_encoded.drop(columns=['Fertilizer_Name'])
y_raw = df_encoded['Fertilizer_Name']

le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

std_preds = prod_model.predict(X_test_sc)
print(f'Standard Test Accuracy: {accuracy_score(y_test, std_preds)*100:.2f}%')
print(f'Standard Weighted F1:    {f1_score(y_test, std_preds, average="weighted"):.4f}')


Standard Test Accuracy: 100.00%
Standard Weighted F1:    1.0000


C:\Users\Chanduru Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


## 4. Robustness Stress Testing with Input Measurement Noise

In [4]:
def perturb_dataset(X_df, seed=42, pct_level=0.05):
    np.random.seed(seed)
    X_pert = X_df.copy()
    for col in ['Nitrogen_Level', 'Phosphorus_Level', 'Potassium_Level']:
        if col in X_pert.columns:
            X_pert[col] = X_pert[col] * (1 + np.random.uniform(-pct_level, pct_level, size=len(X_pert)))
    if 'Soil_pH' in X_pert.columns:
        X_pert['Soil_pH'] = np.clip(X_pert['Soil_pH'] + np.random.uniform(-0.1, 0.1, size=len(X_pert)), 3.5, 9.5)
    return X_pert

seeds = [42, 123, 2026, 999, 7]
p_accs = []
for s in seeds:
    X_p = perturb_dataset(X_test, seed=s, pct_level=0.05)
    X_p_sc = scaler.transform(X_p)
    p_accs.append(accuracy_score(y_test, prod_model.predict(X_p_sc)))

print(f'Robust Mean Accuracy (5% noise): {np.mean(p_accs)*100:.2f}% ± {np.std(p_accs)*100:.2f}%')


Robust Mean Accuracy (5% noise): 95.64% ± 0.27%


C:\Users\Chanduru Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
C:\Users\Chanduru Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
C:\Users\Chanduru Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
C:\Users\Chanduru Kumar\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
C:\Users\Chanduru Kumar\AppData\Local\Pr